In [1]:
import numpy as np
import matplotlib.pyplot as plt

def log_target(x):
    # Log-density of N(0, 1) up to a constant
    return -0.5 * x**2

def rwmh_standard_normal(n_iter=10_000, step_size=1.0, x0=0.0, seed=42):
    rng = np.random.default_rng(seed)
    samples = np.empty(n_iter)
    x = x0
    accepted = 0

    for i in range(n_iter):
        proposal = x + rng.normal(0, step_size)
        log_alpha = log_target(proposal) - log_target(x)
        if np.log(rng.uniform()) < log_alpha:
            x = proposal
            accepted += 1
        samples[i] = x

    return samples, accepted / n_iter

samples, acc_rate = rwmh_standard_normal(n_iter=10_000, step_size=1.5)
burn_in = 1000
post_samples = samples[burn_in:]

print(f"Acceptance Rate: {acc_rate:.3f}")
print(f"Sample Mean: {post_samples.mean():.4f} (True: 0)")
print(f"Sample Std:  {post_samples.std():.4f} (True: 1)")

Acceptance Rate: 0.594
Sample Mean: 0.0186 (True: 0)
Sample Std:  1.0371 (True: 1)


In [3]:
import pymc as pm
import arviz as az

with pm.Model() as bernoulli_model:
    p = pm.Beta("p", alpha=2, beta=2)
    y = pm.Binomial("y", n=12, p=p, observed=8)
    trace_bern = pm.sample(2000, tune=1000, chains=2, random_seed=42, progressbar=False)

print(az.summary(trace_bern, var_names=["p"]))

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [p]
Sampling 2 chains for 1_000 tune and 2_000 draw iterations (2_000 + 4_000 draws total) took 5 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


    mean    sd eti89_lb eti89_ub ess_bulk ess_tail r_hat mcse_mean mcse_sd
p  0.628  0.12     0.42     0.81     1382     2114  1.00    0.0032  0.0017


In [5]:
""" -Trace plot: A time-series plot of sampled parameter values across iterations. Good mixing looks like a stationary "fuzzy caterpillar" with no trends or stuck regions.
-ACF (Autocorrelation Function): Measures the correlation between samples separated by lag $k$. Rapidly decaying autocorrelation indicates efficient exploration; slow decay indicates high dependence between draws.
$\hat{R}$ (Gelman-Rubin statistic): Compares variance between multiple chains to variance within each chain. Values close to $1.00$ (typically $< 1.01$) indicate convergence.ESS (Effective Sample Size): Estimates how many independent draws the correlated MCMC chain is equivalent to. Higher ESS means more precise posterior estimates. """

' -Trace plot: A time-series plot of sampled parameter values across iterations. Good mixing looks like a stationary "fuzzy caterpillar" with no trends or stuck regions.\n-ACF (Autocorrelation Function): Measures the correlation between samples separated by lag $k$. Rapidly decaying autocorrelation indicates efficient exploration; slow decay indicates high dependence between draws.\n$\\hat{R}$ (Gelman-Rubin statistic): Compares variance between multiple chains to variance within each chain. Values close to $1.00$ (typically $< 1.01$) indicate convergence.ESS (Effective Sample Size): Estimates how many independent draws the correlated MCMC chain is equivalent to. Higher ESS means more precise posterior estimates. '

In [6]:
# Generate synthetic normal observations
rng = np.random.default_rng(42)
obs_data = rng.normal(loc=5.0, scale=2.0, size=100)

with pm.Model() as normal_model:
    mu = pm.Normal("mu", mu=0, sigma=10)
    sigma = pm.HalfNormal("sigma", sigma=5)
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=obs_data)

    trace_norm = pm.sample(1000, tune=1000, chains=2, random_seed=42, progressbar=False)

print(az.summary(trace_norm, var_names=["mu", "sigma"]))

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [mu, sigma]
Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 6 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


        mean     sd eti89_lb eti89_ub ess_bulk ess_tail r_hat mcse_mean mcse_sd
mu     4.895  0.152      4.6      5.1     1965     1487  1.00    0.0034  0.0033
sigma  1.572  0.116      1.4      1.8     1702     1237  1.00    0.0028  0.0029
